In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

In [5]:

# ============================================================
# CONFIGURACIÓN
# ============================================================

INPUT_PATH = Path("data/processed/race_laps_2024.parquet")
OUTPUT_PATH = Path("data/processed/race_features_2024.parquet")


# ============================================================
# CARGA
# ============================================================

def load_data():
    print("=" * 70)
    print("CARGANDO DATASET DE CARRERA")
    print("=" * 70)

    df = pd.read_parquet(INPUT_PATH)

    print(f"Filas: {len(df):,}")
    print(f"Columnas: {len(df.columns)}")

    return df


# ============================================================
# LIMPIEZA BÁSICA
# ============================================================

def prepare_data(df):

    df = df.copy()

    # Aseguramos tipos temporales
    if "LapStartTime" in df.columns:
        df["LapStartTime"] = pd.to_timedelta(
            df["LapStartTime"],
            errors="coerce"
        )

    # Orden fundamental
    sort_columns = [
        "Race",
        "LapNumber",
        "Position"
    ]

    df = df.sort_values(sort_columns).reset_index(drop=True)

    return df


# ============================================================
# INFORMACIÓN DE CARRERA
# ============================================================

def add_race_features(df):

    df = df.copy()

    # Número total de vueltas de cada carrera
    race_distance = (
        df.groupby("Race")["LapNumber"]
        .max()
        .rename("RaceDistance")
    )

    df = df.merge(
        race_distance,
        on="Race",
        how="left"
    )

    # Vueltas restantes
    df["LapsRemaining"] = (
        df["RaceDistance"] - df["LapNumber"]
    ).clip(lower=0)

    # --------------------------------------------------------
    # FuelProxy
    # --------------------------------------------------------
    #
    # 1 = inicio de carrera
    # 0 = final de carrera
    #
    # NO representa kg reales de combustible.
    #

    denominator = (df["RaceDistance"] - 1).replace(0, np.nan)

    df["FuelProxy"] = (
        1
        - (df["LapNumber"] - 1) / denominator
    )

    df["FuelProxy"] = df["FuelProxy"].clip(0, 1)

    return df


# ============================================================
# EVOLUCIÓN DE PISTA
# ============================================================

def add_track_evolution(df):

    df = df.copy()

    # Proxy temporal de evolución de pista.
    #
    # 0 = inicio
    # 1 = final
    #
    # En una futura versión lo sustituiremos por una medida
    # basada en el ritmo global de los coches.

    denominator = (df["RaceDistance"] - 1).replace(0, np.nan)

    df["TrackEvolutionProxy"] = (
        (df["LapNumber"] - 1) / denominator
    )

    df["TrackEvolutionProxy"] = (
        df["TrackEvolutionProxy"]
        .clip(0, 1)
    )

    return df


# ============================================================
# TYRELIFE
# ============================================================

def add_tyre_features(df):

    df = df.copy()

    # --------------------------------------------------------
    # Duración observada del stint
    # --------------------------------------------------------

    stint_length = (
        df.groupby(
            ["Race", "Driver", "Stint"]
        )["TyreLife"]
        .transform("max")
    )

    denominator = (stint_length - 1).replace(0, np.nan)

    df["TyreLifeNormalized"] = (
        (df["TyreLife"] - 1) / denominator
    )

    df["TyreLifeNormalized"] = (
        df["TyreLifeNormalized"]
        .clip(0, 1)
    )

    return df


# ============================================================
# COCHE DE DELANTE / DETRÁS
# ============================================================

def add_driver_gaps(df):

    df = df.copy()

    # --------------------------------------------------------
    # Ordenamos por posición dentro de cada vuelta
    # --------------------------------------------------------

    df = df.sort_values(
        ["Race", "LapNumber", "Position"]
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Piloto delante
    # --------------------------------------------------------

    df["DriverAhead"] = (
        df.groupby(
            ["Race", "LapNumber"]
        )["Driver"]
        .shift(1)
    )

    # --------------------------------------------------------
    # Piloto detrás
    # --------------------------------------------------------

    df["DriverBehind"] = (
        df.groupby(
            ["Race", "LapNumber"]
        )["Driver"]
        .shift(-1)
    )

    # --------------------------------------------------------
    # Tiempo de inicio de vuelta del coche delante
    # --------------------------------------------------------

    df["LapStartTimeAhead"] = (
        df.groupby(
            ["Race", "LapNumber"]
        )["LapStartTime"]
        .shift(1)
    )

    # --------------------------------------------------------
    # Tiempo de inicio de vuelta del coche detrás
    # --------------------------------------------------------

    df["LapStartTimeBehind"] = (
        df.groupby(
            ["Race", "LapNumber"]
        )["LapStartTime"]
        .shift(-1)
    )

    # --------------------------------------------------------
    # GAP DELANTE
    # --------------------------------------------------------

    df["GapAheadProxy"] = (
        df["LapStartTime"]
        - df["LapStartTimeAhead"]
    ).dt.total_seconds()

    # --------------------------------------------------------
    # GAP DETRÁS
    # --------------------------------------------------------

    df["GapBehindProxy"] = (
        df["LapStartTimeBehind"]
        - df["LapStartTime"]
    ).dt.total_seconds()

    # --------------------------------------------------------
    # Valores imposibles
    # --------------------------------------------------------

    df.loc[
        df["GapAheadProxy"] < 0,
        "GapAheadProxy"
    ] = np.nan

    df.loc[
        df["GapBehindProxy"] < 0,
        "GapBehindProxy"
    ] = np.nan

    # --------------------------------------------------------
    # Tendencia del gap
    # --------------------------------------------------------

    df["GapAheadTrend"] = (
        df.groupby(
            ["Race", "Driver"]
        )["GapAheadProxy"]
        .diff()
    )

    # --------------------------------------------------------
    # TRÁFICO
    # --------------------------------------------------------

    df["InTraffic"] = (
        df["GapAheadProxy"] < 1.0
    )

    # Si no existe gap, no podemos afirmar que hay tráfico
    df.loc[
        df["GapAheadProxy"].isna(),
        "InTraffic"
    ] = False

    # --------------------------------------------------------
    # POTENCIAL DRS
    # --------------------------------------------------------

    df["PotentialDRS"] = (
        df["GapAheadProxy"] < 1.0
    )

    df.loc[
        df["GapAheadProxy"].isna(),
        "PotentialDRS"
    ] = False

    return df


# ============================================================
# TRACK STATUS
# ============================================================

def add_track_status_features(df):

    df = df.copy()

    # Primero inspeccionaremos los valores reales de TrackStatus.
    print("\nValores de TrackStatus:")
    print(df["TrackStatus"].value_counts(dropna=False).sort_index())

    # Dejamos el valor original.
    # No inventamos todavía el significado de cada código.

    return df


# ============================================================
# SELECCIÓN FINAL
# ============================================================

def select_features(df):

    feature_columns = [
        # Identificación
        "Race",
        "Driver",
        "DriverNumber",
        "Team",

        # Carrera
        "LapNumber",
        "Position",

        # Neumáticos
        "Compound",
        "TyreLife",
        "TyreLifeNormalized",
        "FreshTyre",
        "Stint",

        # Combustible
        "LapsRemaining",
        "FuelProxy",

        # Evolución pista
        "TrackEvolutionProxy",

        # Tráfico
        "DriverAhead",
        "DriverBehind",
        "GapAheadProxy",
        "GapBehindProxy",
        "GapAheadTrend",
        "InTraffic",
        "PotentialDRS",

        # Estado de pista
        "TrackStatus",

        # Target
        "LapTimeSeconds",
    ]

    # Nos quedamos únicamente con columnas existentes
    feature_columns = [
        col for col in feature_columns
        if col in df.columns
    ]

    return df[feature_columns].copy()


# ============================================================
# INSPECCIÓN
# ============================================================

def inspect_features(df):

    print("\n" + "=" * 70)
    print("INSPECCIÓN DEL FEATURE DATASET")
    print("=" * 70)

    print(f"\nFilas: {len(df):,}")
    print(f"Columnas: {len(df.columns)}")

    print("\nColumnas:")
    print(df.columns.tolist())

    print("\nValores nulos:")
    print(
        df.isna()
        .sum()
        .sort_values(ascending=False)
    )

    print("\nGapAheadProxy:")
    print(
        df["GapAheadProxy"]
        .describe()
    )

    print("\nGapBehindProxy:")
    print(
        df["GapBehindProxy"]
        .describe()
    )

    print("\nInTraffic:")
    print(
        df["InTraffic"]
        .value_counts(dropna=False)
    )

    print("\nPotentialDRS:")
    print(
        df["PotentialDRS"]
        .value_counts(dropna=False)
    )

    print("\nFuelProxy:")
    print(
        df["FuelProxy"]
        .describe()
    )

    print("\nTrackEvolutionProxy:")
    print(
        df["TrackEvolutionProxy"]
        .describe()
    )

    print("\nEjemplos:")
    print(
        df[
            [
                "Race",
                "LapNumber",
                "Driver",
                "Position",
                "DriverAhead",
                "GapAheadProxy",
                "GapAheadTrend",
                "InTraffic",
                "PotentialDRS",
                "FuelProxy",
                "TrackEvolutionProxy",
                "TyreLife",
                "Compound",
                "LapTimeSeconds",
            ]
        ].head(20).to_string(index=False)
    )


# ============================================================
# MAIN
# ============================================================

def main():

    df = load_data()

    df = prepare_data(df)

    df = add_race_features(df)

    df = add_track_evolution(df)

    df = add_tyre_features(df)

    df = add_driver_gaps(df)

    df = add_track_status_features(df)

    df = select_features(df)

    inspect_features(df)

    OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_parquet(
        OUTPUT_PATH,
        index=False
    )

    print("\n" + "=" * 70)
    print("FEATURE DATASET CREADO")
    print("=" * 70)

    print(f"Guardado en: {OUTPUT_PATH}")
    print(f"Filas: {len(df):,}")
    print(f"Columnas: {len(df.columns)}")


